<a href="https://colab.research.google.com/github/SubeenJung0228/Problem-Set-econ-3916/blob/main/Problem%20Set%203/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [3]:
# Customers. Draws in a fixed order (region, n_orders, revenue, signup_date) so the grader can rerun this.
region = rng.choice(["West", "East"], size=N_CUSTOMERS, p=[0.7, 0.3])          # about 2,100 West
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)            # mean near 7, long right tail
revenue = np.maximum(REV_PER_ORDER * n_orders
                     + rng.normal(REV_MEAN, REV_SD, N_CUSTOMERS), REV_FLOOR).round(2)   # driven by order count
signup_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(rng.integers(0, 3 * 365, N_CUSTOMERS), unit="D")

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "region": region,
    "revenue": revenue,
    "signup_date": signup_date,
})

# Orders: one row for each of a customer's n_orders
orders_df = pd.DataFrame({"customer_id": np.repeat(customers_df["customer_id"].to_numpy(), n_orders)})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

assert len(customers_df) >= N_CUSTOMERS
assert (customers_df["region"] == "West").sum() >= MIN_WEST, "fewer than 2,000 West customers"

# Register both with DuckDB and confirm they can be queried
con.register("customers_df", customers_df)
con.register("orders_df", orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

print(con.sql("""SELECT region, COUNT(*) AS customers, ROUND(AVG(revenue), 2) AS avg_revenue,
                        MIN(signup_date) AS first_signup, MAX(signup_date) AS last_signup
                 FROM customers GROUP BY region ORDER BY region"""))
print(con.sql("""SELECT COUNT(*) AS orders, COUNT(DISTINCT customer_id) AS customers_with_orders,
                        ROUND(COUNT(*) / COUNT(DISTINCT customer_id), 2) AS orders_per_customer,
                        MAX(n) AS max_orders
                 FROM (SELECT customer_id, COUNT(*) AS n FROM orders GROUP BY customer_id) t
                 JOIN orders USING (customer_id)"""))

┌─────────┬───────────┬─────────────┬─────────────────────┬─────────────────────┐
│ region  │ customers │ avg_revenue │    first_signup     │     last_signup     │
│ varchar │   int64   │   double    │    timestamp_ns     │    timestamp_ns     │
├─────────┼───────────┼─────────────┼─────────────────────┼─────────────────────┤
│ East    │       903 │      818.41 │ 2022-01-01 00:00:00 │ 2024-12-30 00:00:00 │
│ West    │      2097 │      814.43 │ 2022-01-01 00:00:00 │ 2024-12-30 00:00:00 │
└─────────┴───────────┴─────────────┴─────────────────────┴─────────────────────┘

┌────────┬───────────────────────┬─────────────────────┬────────────┐
│ orders │ customers_with_orders │ orders_per_customer │ max_orders │
│ int64  │         int64         │       double        │   int64    │
├────────┼───────────────────────┼─────────────────────┼────────────┤
│  20584 │                  3000 │                6.86 │         57 │
└────────┴───────────────────────┴─────────────────────┴────────────┘



### Step 1.2: Write both queries (7 pts)

In [4]:
PLAIN_SQL = """SELECT customer_id, revenue
               FROM customers
               WHERE region = 'West'"""

JOINED_SQL = """SELECT c.customer_id, c.revenue
                FROM customers c
                JOIN orders o ON o.customer_id = c.customer_id
                WHERE c.region = 'West'"""

plain_avg, plain_rows = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({PLAIN_SQL})").fetchone()
joined_avg, joined_rows = con.execute(f"SELECT AVG(revenue), COUNT(*) FROM ({JOINED_SQL})").fetchone()

print(f"West, un-joined: ${plain_avg:,.2f} over {plain_rows:,} rows")
print(f"West, joined:    ${joined_avg:,.2f} over {joined_rows:,} rows")
print(f"ratio:           {joined_avg / plain_avg:.2f}")

West, un-joined: $814.43 over 2,097 rows
West, joined:    $1,250.67 over 14,191 rows
ratio:           1.54


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*

Both queries run, but only the un-joined query is correct. It calculates the average over 2,097 West customers and gives \$814.43. The joined query uses 14,191 orders and gives $1,250.67.

The correct answer is $814.43. Revenue is measured per customer, so each customer must count once. The join repeats each customer's revenue for every order, which gives more weight to customers with more orders and produces an order-weighted average instead of a customer average. The key is to match the unit of observation to the question.

### Step 1.3: The rule (10 pts)

In [5]:
def join_check(con, base_sql, joined_sql):
    """Compare the row count before and after a join.
    Returns a warning string if the join changed the row count, else None."""
    n_before = con.execute(f"SELECT COUNT(*) FROM ({base_sql})").fetchone()[0]
    n_after = con.execute(f"SELECT COUNT(*) FROM ({joined_sql})").fetchone()[0]
    if n_after != n_before:
        return (f"WARNING: the join changed the row count from {n_before:,} to {n_after:,} "
                f"({n_after / n_before:.2f}x). Each row is no longer one unit of the base table; "
                f"aggregate the right-hand table to the base table's grain before joining.")
    return None

GOOD_SQL = """WITH order_counts AS (
                  SELECT customer_id, COUNT(*) AS n_orders
                  FROM orders
                  GROUP BY customer_id
              )
              SELECT c.customer_id, c.revenue, oc.n_orders
              FROM customers c
              JOIN order_counts oc ON oc.customer_id = c.customer_id
              WHERE c.region = 'West'"""

bad_warning = join_check(con, PLAIN_SQL, JOINED_SQL)
good_warning = join_check(con, PLAIN_SQL, GOOD_SQL)

print("bad query: ", bad_warning)
print("good query:", good_warning)

good_avg = con.execute(f"SELECT AVG(revenue) FROM ({GOOD_SQL})").fetchone()[0]
print(f"\ngood query's average: ${good_avg:,.2f}  (un-joined: ${plain_avg:,.2f})")

bad query:  WARNING: the join changed the row count from 2,097 to 14,191 (6.77x). Each row is no longer one unit of the base table; aggregate the right-hand table to the base table's grain before joining.
good query: None

good query's average: $814.43  (un-joined: $814.43)


*Your answer here.*

The key rule is that a join should not change the row count of the table whose column you are averaging. join_check compares the row counts before and after a join. In the bad query, the count increases from 2,097 customers to 14,191 orders, meaning each customer is counted multiple times. This makes the average weighted by the number of orders. In the good query, the order_counts CTE first groups orders by customer, so the join remains one-to-one. The row count stays at 2,097, and the average remains $814.43.

This check only requires two COUNT(*) queries and helps catch changes in the unit of observation before reporting results. A change in row count is not always wrong, but it should always be intentional rather than accidental.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [6]:
unknown_ids = rng.choice(customers_df["customer_id"].to_numpy(), size=450, replace=False)
con.register("unknown_ids", pd.DataFrame({"customer_id": unknown_ids}))
con.execute("UPDATE customers SET region = NULL WHERE customer_id IN (SELECT customer_id FROM unknown_ids)")

print(con.sql("""SELECT region, COUNT(*) AS customers
                 FROM customers GROUP BY region ORDER BY region NULLS LAST"""))

┌─────────┬───────────┐
│ region  │ customers │
│ varchar │   int64   │
├─────────┼───────────┤
│ East    │       760 │
│ West    │      1790 │
│ NULL    │       450 │
└─────────┴───────────┘



### Step 2.2: Lose them, then find them (15 pts)

In [7]:
# What the analyst ran: "everyone outside the West"
n_ne = con.execute("SELECT COUNT(*) FROM customers WHERE region != 'West'").fetchone()[0]

# What the analyst meant: every customer not known to be in the West, including unknown region
n_meant = con.execute("SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'").fetchone()[0]

n_null = con.execute("SELECT COUNT(*) FROM customers WHERE region IS NULL").fetchone()[0]

print(f"WHERE region != 'West':                {n_ne:,} rows")
print(f"WHERE region IS DISTINCT FROM 'West':  {n_meant:,} rows")
print(f"difference: {n_meant - n_ne:,} rows  (customers with a NULL region: {n_null:,})")
print(con.sql("SELECT NULL != 'West' AS null_ne, NULL IS DISTINCT FROM 'West' AS null_distinct"))

WHERE region != 'West':                760 rows
WHERE region IS DISTINCT FROM 'West':  1,210 rows
difference: 450 rows  (customers with a NULL region: 450)
┌─────────┬───────────────┐
│ null_ne │ null_distinct │
│ boolean │    boolean    │
├─────────┼───────────────┤
│ NULL    │ true          │
└─────────┴───────────────┘



*Your answer here.*

The key difference is that != 'West' excludes customers with NULL regions, while IS DISTINCT FROM 'West' includes them. The first query returns 760 rows, but the second returns 1,210, a difference of 450 customers.

This happens because region != 'West' evaluates to NULL when the region is missing. Since WHERE keeps only rows where the condition is TRUE, those customers are silently dropped.

This can bias regional comparisons if customers with missing regions differ systematically from others. For example, if newer customers are more likely to have missing regions, excluding them could make retention or churn rates look better than they actually are. The best approach is to decide explicitly how to handle NULL values, either include them with IS DISTINCT FROM or report them as a separate "Unknown" group. Also, report how many customers are excluded from each comparison.

### Step 2.3: Aggregate without collapsing (10 pts)

In [8]:
WINDOW_SQL = """
    SELECT customer_id,
           region,
           revenue,
           AVG(revenue) OVER (PARTITION BY region)           AS region_avg,
           revenue - AVG(revenue) OVER (PARTITION BY region) AS vs_region,
           CASE WHEN revenue >= AVG(revenue) OVER (PARTITION BY region)
                THEN 'above' ELSE 'below' END                AS position
    FROM customers
    ORDER BY customer_id
"""

with_region_avg = con.execute(WINDOW_SQL).df()

n_customers = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
print(with_region_avg.head(8).round(2).to_string(index=False))
print(f"\nrows in {n_customers:,}, rows out {len(with_region_avg):,}")
assert len(with_region_avg) == n_customers, "the row count changed"

 customer_id region  revenue  region_avg  vs_region position
           1   East  1091.60      803.64     287.96    above
           2   West   791.79      813.37     -21.58    below
           3   East   803.03      803.64      -0.61    below
           4   West   503.94      813.37    -309.43    below
           5   None  1411.17      844.85     566.32    above
           6   East   381.69      803.64    -421.95    below
           7   East  2738.42      803.64    1934.78    above
           8   East   554.64      803.64    -249.00    below

rows in 3,000, rows out 3,000


AVG(revenue) OVER (PARTITION BY region) calculates each region’s average without collapsing the customer rows. Each customer keeps their own revenue and gets their region’s average, the difference, and an indicator of whether they are above or below average. The row count stays at 3,000, which the assert confirms.

Unlike GROUP BY, which returns one row per region, a window function preserves every customer row.

One important detail is that PARTITION BY groups all NULL regions together. Therefore, the 450 customers with unknown regions are compared against each other, using their group’s average of $844.85. They are neither dropped nor mixed with East or West. The director should understand that, for these customers, “their region’s average” means the average among customers whose region is also unknown.

## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [9]:
TRUE_EFFECT = 25.0           # the programme adds $25 of revenue: small, and known
n_sim = 5_000

account_size = rng.normal(100, 30, n_sim)                                    # the confounder
west = (rng.random(n_sim) < 1 / (1 + np.exp(-(account_size - 100) / 15))).astype(int)   # bigger accounts were assigned to West
revenue = 200 + 6 * account_size + TRUE_EFFECT * west + rng.normal(0, 80, n_sim)         # bigger accounts also earn more

sim = pd.DataFrame({"account_size": account_size, "west": west, "revenue": revenue})

naive = sim.loc[sim.west == 1, "revenue"].mean() - sim.loc[sim.west == 0, "revenue"].mean()
adjusted = smf.ols("revenue ~ west + account_size", data=sim).fit().params["west"]

print(f"truth ${TRUE_EFFECT:.2f} | naive ${naive:.2f} | adjusted for account size ${adjusted:.2f}")
print(f"mean account size: West {sim.loc[sim.west == 1, 'account_size'].mean():.1f}, "
      f"elsewhere {sim.loc[sim.west == 0, 'account_size'].mean():.1f}")

truth $25.00 | naive $250.49 | adjusted for account size $25.43
mean account size: West 118.8, elsewhere 81.3


*Your answer here.*

The simulation shows how selection bias can exaggerate a programme’s effect. Larger accounts were more likely to be assigned to the West and also generated more revenue, with each unit of account size adding \$6. The programme itself increased revenue by only \$25.

The naive comparison shows a difference of \$250.49, which is about ten times the true effect. This is because it attributes the higher revenue of larger accounts to the programme. After controlling for account size in a regression, the estimated effect drops to \$25.43, which is very close to the true $25.

The main takeaway is that higher revenue in the West does not necessarily mean the programme is effective. West and non-West customers were already different before the programme began. Unless account size is controlled for, the effect of those pre-existing differences can be mistaken for the programme's effect.

### Step 3.2: The control that ruins it (13 pts)

In [10]:
tickets = rng.poisson(np.maximum(0, 2 + 4 * sim["west"] + 0.02 * (sim["revenue"] - 800)))
sim["tickets"] = tickets

with_tickets = smf.ols("revenue ~ west + account_size + tickets", data=sim).fit().params["west"]

print(f"truth ${TRUE_EFFECT:.2f} | naive ${naive:.2f} | adjusted ${adjusted:.2f} | adding tickets ${with_tickets:.2f}")
print(f"mean tickets: West {sim.loc[sim.west == 1, 'tickets'].mean():.1f}, "
      f"elsewhere {sim.loc[sim.west == 0, 'tickets'].mean():.1f}")

truth $25.00 | naive $250.49 | adjusted $25.43 | adding tickets $-28.84
mean tickets: West 8.7, elsewhere 1.2


*Your answer here.*

Adding support tickets as a control made the estimate worse, changing it from \$25.43 to -$28.84. This happens because tickets are affected by both the programme and revenue. Programme customers receive extra tickets through their account managers, while higher-spending customers also tend to contact support more. Comparing customers with the same number of tickets therefore creates an unfair comparison. West customers may have lower revenue than non-West customers with the same ticket count because some of their tickets came from the programme rather than spending. This makes the programme look less effective than it really is. Since tickets are influenced by the programme, they should not be controlled for. We should instead control for factors that existed before the programme, such as account size.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [11]:
"""
query_review.py: pre-dashboard SQL review for DuckDB.

    warnings = review_query(con, sql)
    warnings_to_frame(warnings)   # optional: a pandas DataFrame for eyeballing

Dependencies: duckdb and pandas only. The SQL is parsed by DuckDB itself via
json_serialize_sql(), so there is no third-party parser and no chance that
the reviewer reads the query differently from the engine that will run it.

Checks (warning codes are stable so tests can assert on them):
    JOIN_FANOUT          a join repeats rows of the driving table (row count grows)
    JOIN_DROPS_ROWS      an INNER JOIN removes unmatched rows (row count shrinks)
    NULL_FILTER          a WHERE predicate silently drops rows whose column is NULL
    LEFT_JOIN_NULLIFIED  a WHERE predicate on the optional side of an outer join
    NOT_IN_WITH_NULL     NOT IN (..., NULL, ...) can never be TRUE
    AGG_OVER_FANOUT      SUM/AVG/COUNT/... over rows that a join has repeated
    NULL_FILTER_LATENT   (opt-in) filter on a nullable column that has no NULLs today
    UNREVIEWABLE         something we could not analyse; never silently skipped
    UNPARSEABLE          DuckDB could not parse the statement as a SELECT
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from typing import Any, Iterator

import duckdb
import pandas as pd

def _s(v):
    """Return v if it is a string, else '' (some AST 'type' fields are nested dicts)."""
    return v if isinstance(v, str) else ""

ERROR, WARN, INFO = "error", "warn", "info"
_SEVERITY_RANK = {ERROR: 0, WARN: 1, INFO: 2}


@dataclass
class ReviewWarning:
    code: str
    severity: str
    message: str
    detail: dict[str, Any] = field(default_factory=dict)


def warnings_to_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    return pd.DataFrame([asdict(w) for w in warnings],
                        columns=["code", "severity", "message", "detail"])


# --------------------------------------------------------------------------
# Public entry point
# --------------------------------------------------------------------------
def review_query(con: duckdb.DuckDBPyConnection, sql: str, *,
                 include_latent_nulls: bool = False) -> list[ReviewWarning]:
    """Return structured warnings for a single SELECT. Never executes `sql`
    itself; only small profiling queries against the tables it reads."""
    try:
        raw = con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0]
        tree = json.loads(raw)
    except duckdb.Error as exc:
        return [ReviewWarning("UNPARSEABLE", ERROR, f"DuckDB could not parse the query: {exc}")]
    if tree.get("error"):
        return [ReviewWarning("UNPARSEABLE", ERROR,
                              f"DuckDB could not parse the query as a SELECT: "
                              f"{tree.get('error_message', 'unknown error')}")]
    statements = tree.get("statements", [])

    # CTE name -> its SELECT statement, so `FROM my_cte` can be profiled.
    ctes: dict[str, dict] = {}
    for node in _walk(statements):
        for entry in (node.get("cte_map") or {}).get("map", []) or []:
            query = (entry.get("value") or {}).get("query")
            if entry.get("key") and query:
                ctes[entry["key"].lower()] = query

    # Every SELECT_NODE (outer query, CTE bodies, subqueries) is its own scope.
    warnings: list[ReviewWarning] = []
    for node in _walk(statements):
        if node.get("type") == "SELECT_NODE":
            warnings += _Scope(con, node, ctes, include_latent_nulls).review()

    unique, seen = [], set()
    for w in warnings:
        if (w.code, w.message) not in seen:
            seen.add((w.code, w.message))
            unique.append(w)
    return sorted(unique, key=lambda w: _SEVERITY_RANK[w.severity])


# --------------------------------------------------------------------------
# AST helpers (all knowledge of DuckDB's JSON AST shape lives in this file)
# --------------------------------------------------------------------------
_OP_SYMBOL = {
    "COMPARE_EQUAL": "=", "COMPARE_NOTEQUAL": "<>",
    "COMPARE_LESSTHAN": "<", "COMPARE_GREATERTHAN": ">",
    "COMPARE_LESSTHANOREQUALTO": "<=", "COMPARE_GREATERTHANOREQUALTO": ">=",
    "COMPARE_IN": "IN", "COMPARE_NOT_IN": "NOT IN",
    "COMPARE_BETWEEN": "BETWEEN", "COMPARE_NOT_BETWEEN": "NOT BETWEEN",
}
_NEGATING_OPS = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN", "COMPARE_NOT_BETWEEN"}
_LIKE_FUNCS = {"~~": "LIKE", "!~~": "NOT LIKE", "~~*": "ILIKE", "!~~*": "NOT ILIKE",
               "like": "LIKE", "not_like": "NOT LIKE", "ilike": "ILIKE",
               "not_ilike": "NOT ILIKE", "regexp_matches": "regexp_matches",
               "contains": "contains", "starts_with": "starts_with",
               "prefix": "prefix", "suffix": "suffix"}
_GUARD_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE",
                "COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
_GUARD_FUNCS = {"coalesce", "ifnull", "nvl"}

# Aggregates whose result changes when input rows are duplicated. MIN, MAX,
# BOOL_AND/OR, ANY_VALUE are idempotent (f(x, x) = f(x)) so are left out.
_FANOUT_SENSITIVE_AGGS = {
    "sum": "inflated", "fsum": "inflated", "sumkahan": "inflated",
    "kahan_sum": "inflated", "product": "inflated",
    "count": "inflated", "count_star": "inflated",
    "avg": "weighted toward keys with more matches",
    "mean": "weighted toward keys with more matches",
    "median": "weighted toward keys with more matches",
    "quantile_cont": "weighted toward keys with more matches",
    "stddev_samp": "distorted", "stddev_pop": "distorted",
    "string_agg": "padded with repeats", "list": "padded with repeats",
    "array_agg": "padded with repeats",
}


def _walk(node: Any) -> Iterator[dict]:
    """Every dict in the tree, including nested queries."""
    if isinstance(node, dict):
        yield node
        for v in node.values():
            yield from _walk(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk(v)


def _walk_expr(node: Any) -> Iterator[dict]:
    """Every expression dict, without descending into subqueries; those have
    their own scope and are reviewed as their own SELECT_NODE."""
    if isinstance(node, dict):
        yield node
        if (node.get("class") or "").upper() == "SUBQUERY":
            yield from _walk_expr(node.get("child"))
            return
        for v in node.values():
            yield from _walk_expr(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk_expr(v)


def _colrefs(expr: Any) -> list[dict]:
    return [e for e in _walk_expr(expr) if (e.get("class") or "").upper() == "COLUMN_REF"]


def _conjuncts(expr: dict | None) -> list[dict]:
    if not expr:
        return []
    if (expr.get("type") or "").upper() == "CONJUNCTION_AND":
        return [x for c in expr.get("children", []) for x in _conjuncts(c)]
    return [expr]


def _predicates(e: Any, negated: bool = False) -> Iterator[tuple[dict, bool, str]]:
    """Yield (predicate, effectively_negated, display_operator) for every
    null-sensitive leaf predicate, tracking NOT through AND/OR."""
    if not isinstance(e, dict):
        return
    t, cls = (e.get("type") or "").upper(), (e.get("class") or "").upper()
    if t == "OPERATOR_NOT":
        for c in e.get("children", []):
            yield from _predicates(c, not negated)
    elif t in ("CONJUNCTION_AND", "CONJUNCTION_OR"):
        for c in e.get("children", []):
            yield from _predicates(c, negated)
    elif cls == "SUBQUERY":
        if (e.get("subquery_type") or "").upper() == "ANY" and e.get("child"):
            yield e, negated, "NOT IN (subquery)" if negated else "IN (subquery)"
    elif t in _OP_SYMBOL:
        sym = _OP_SYMBOL[t]
        yield e, negated ^ (t in _NEGATING_OPS), f"NOT {sym}" if negated else sym
    elif cls == "FUNCTION" and (e.get("function_name") or "").lower() in _LIKE_FUNCS:
        name = _LIKE_FUNCS[e["function_name"].lower()]
        yield e, negated ^ name.startswith("NOT"), f"NOT {name}" if negated else name


def _is_column_equality(e: dict) -> bool:
    return ((e.get("type") or "").upper() == "COMPARE_EQUAL"
            and (e.get("left") or {}).get("class") == "COLUMN_REF"
            and (e.get("right") or {}).get("class") == "COLUMN_REF")


def _not_in_with_null(pred: dict, negated: bool) -> bool:
    if not negated or (pred.get("type") or "").upper() not in ("COMPARE_IN", "COMPARE_NOT_IN"):
        return False
    return any((c.get("class") or "").upper() == "CONSTANT" and (c.get("value") or {}).get("is_null")
               for c in (pred.get("children") or [])[1:])


def _q(ident: str) -> str:
    return '"' + ident.replace('"', '""') + '"'


# --------------------------------------------------------------------------
# One SELECT scope
# --------------------------------------------------------------------------
@dataclass
class _Rel:
    alias: str
    label: str
    sql: str | None            # SQL producing this relation's rows; None = cannot profile
    is_table: bool
    columns: dict[str, bool] | None = None   # lower(name) -> declared nullable

    def from_sql(self) -> str:
        return self.sql if self.is_table else f"({self.sql})"


@dataclass
class _Join:
    join_type: str
    ref_type: str
    left: set[str]
    right: set[str]
    node: dict
    keys: list[tuple[str, str, str, str]] = field(default_factory=list)  # (l_alias, l_col, r_alias, r_col)


class _Scope:
    def __init__(self, con, node: dict, ctes: dict[str, dict], include_latent: bool):
        self.con, self.node, self.ctes, self.include_latent = con, node, ctes, include_latent
        self.rels: dict[str, _Rel] = {}
        self.joins: list[_Join] = []
        self._nulls: dict[tuple[str, str], tuple[int, int] | None] = {}
        if node.get("from_table"):
            self._flatten(node["from_table"])
        # Comma joins (FROM a, b WHERE a.id = b.id) parse as CROSS joins with
        # the key in WHERE; recover those keys so they aren't mistaken for
        # a cartesian product.
        for j in self.joins:
            if not j.keys and j.node.get("condition") is None:
                for conj in _conjuncts(node.get("where_clause")):
                    k = self._equi_pair(conj, j.left, j.right)
                    if k:
                        j.keys.append(k)

    # ---- relations and column resolution --------------------------------
    def _flatten(self, t: dict) -> set[str]:
        kind = (t.get("type") or "").upper()
        if kind == "JOIN":
            left, right = self._flatten(t["left"]), self._flatten(t["right"])
            j = _Join((t.get("join_type") or "INNER").upper(),
                      (t.get("ref_type") or "REGULAR").upper(), left, right, t)
            for c in t.get("using_columns") or []:
                la, ra = self._owner(c, left), self._owner(c, right)
                if la and ra:
                    j.keys.append((la, c, ra, c))
            for conj in _conjuncts(t.get("condition")):
                k = self._equi_pair(conj, left, right)
                if k:
                    j.keys.append(k)
            self.joins.append(j)
            return left | right
        alias = (t.get("alias") or t.get("table_name") or f"_rel{len(self.rels)}").lower()
        self.rels[alias] = self._make_rel(alias, t, kind)
        return {alias}

    def _make_rel(self, alias: str, t: dict, kind: str) -> _Rel:
        if kind == "BASE_TABLE":
            name = t["table_name"]
            if name.lower() in self.ctes:
                return _Rel(alias, f"CTE {name}", self._to_sql(self.ctes[name.lower()]), False)
            parts = [p for p in (t.get("catalog_name"), t.get("schema_name"), name) if p]
            return _Rel(alias, name, ".".join(_q(p) for p in parts), True)
        if kind == "SUBQUERY":
            return _Rel(alias, f"subquery {alias}", self._to_sql(t.get("subquery")), False)
        return _Rel(alias, f"{kind.lower()} {alias}", None, False)

    def _to_sql(self, stmt: dict | None) -> str | None:
        """Turn a parsed sub-statement back into SQL with DuckDB's own
        json_deserialize_sql, so subqueries and CTEs can be profiled."""
        if not stmt:
            return None
        try:
            payload = json.dumps({"error": False, "statements": [stmt]})
            return self.con.execute("SELECT json_deserialize_sql(?::JSON)", [payload]).fetchone()[0]
        except duckdb.Error:
            return None

    def columns(self, rel: _Rel) -> dict[str, bool]:
        if rel.columns is None:
            rel.columns = {}
            if rel.sql:
                try:
                    df = self.con.execute(f"DESCRIBE {rel.sql}").df()
                    rel.columns = {str(n).lower(): str(nl).upper() != "NO"
                                   for n, nl in zip(df["column_name"], df["null"])}
                except duckdb.Error:
                    pass
        return rel.columns

    def _owner(self, col: str, aliases: set[str]) -> str | None:
        owners = [a for a in aliases if col.lower() in self.columns(self.rels[a])]
        return owners[0] if len(owners) == 1 else None

    def resolve(self, colref: dict) -> tuple[str, str] | None:
        names = colref.get("column_names") or []
        if not names:
            return None
        col = names[-1]
        if len(names) >= 2:
            q = names[-2].lower()
            if q in self.rels:
                return q, col
            for a, r in self.rels.items():          # qualified by table name, not alias
                if r.label.lower() == q:
                    return a, col
            return None                             # correlated / outer reference
        owner = self._owner(col, set(self.rels))
        return (owner, col) if owner else None

    def _equi_pair(self, e: dict, left: set[str], right: set[str]):
        if not _is_column_equality(e):
            return None
        a, b = self.resolve(e["left"]), self.resolve(e["right"])
        if not a or not b:
            return None
        if a[0] in left and b[0] in right:
            return (*a, *b)
        if b[0] in left and a[0] in right:
            return (*b, *a)
        return None

    def _label(self, aliases: set[str]) -> str:
        return ", ".join(self.rels[a].label for a in sorted(aliases))

    # ---- the review ----------------------------------------------------
    def review(self) -> list[ReviewWarning]:
        warnings: list[ReviewWarning] = []
        repeated: dict[str, int] = {}       # alias -> max times a row is repeated by some join
        fanned_out: set[str] = set()        # aliases repeated beyond the driving table's grain
        outer_nullable: dict[str, dict] = {}
        for j in self.joins:
            warnings += self._check_join(j, repeated, fanned_out, outer_nullable)
        warnings += self._check_where(outer_nullable)
        warnings += self._check_aggregates(repeated, fanned_out)
        return warnings

    # ---- check 1: joins that change the row count -------------------------
    #
    # HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (TWICE OR AT ALL)
    #
    # The naive test is COUNT(*) of the query with and without the join,
    # which runs the expensive join and then runs it again for every variant.
    # Instead we use the fact that an equi-join's output size is fully
    # determined by key multiplicities:
    #
    #     |L ⋈ R| = Σ over keys k of  n_L(k) · n_R(k)
    #
    # where n_L(k) is how many rows of L carry key k. So we GROUP BY the join
    # key on each side separately (one scan per table, output size = number
    # of distinct keys), join those two small summaries, and read off:
    #
    #     matched_rows       Σ n_L·n_R                  exact inner-join row count
    #     left_matched       Σ n_L over matched keys    L rows that find a partner
    #     max_right_per_key  max n_R                    >1  ⇒ L rows get repeated
    #     max_left_per_key   max n_L                    >1  ⇒ R rows get repeated
    #
    # LEFT/RIGHT/FULL counts follow by adding back unmatched rows. NULL keys
    # fall out naturally, because the summary join uses `=`, exactly as the
    # real join does. The cost is two GROUP BYs, never the join's output
    # (which is the thing that explodes when there is fan-out).
    #
    # Approximation: for a chain of joins we profile against the base table
    # that owns each key, and over the whole table rather than after the
    # query's WHERE. That can over-report, never under-report, duplication.
    def _check_join(self, j: _Join, repeated, fanned_out, outer_nullable) -> list[ReviewWarning]:
        if j.ref_type in ("ASOF", "POSITIONAL") or j.join_type in ("SEMI", "ANTI"):
            return []      # these joins cannot repeat rows
        label = f"{self._label(j.left)} {j.join_type} JOIN {self._label(j.right)}"
        on = " AND ".join(f"{la}.{lc} = {ra}.{rc}" for la, lc, ra, rc in j.keys) or "(no equi-key)"
        p = self._profile(j, label)
        out: list[ReviewWarning] = []
        if isinstance(p, ReviewWarning):
            out.append(p)
            p = None

        if j.join_type in ("LEFT", "OUTER"):
            for a in j.right:
                outer_nullable[a] = {"join": label,
                                     "unmatched": p and p["left_rows"] - p["left_matched"]}
        if j.join_type in ("RIGHT", "OUTER"):
            for a in j.left:
                outer_nullable[a] = {"join": label,
                                     "unmatched": p and p["right_rows"] - p["right_matched"]}
        if p is None:
            return out

        rows_out = p["matched_rows"]
        if j.join_type in ("LEFT", "OUTER"):
            rows_out += p["left_rows"] - p["left_matched"]
        if j.join_type in ("RIGHT", "OUTER"):
            rows_out += p["right_rows"] - p["right_matched"]
        driving_is_right = j.join_type == "RIGHT"
        rows_in = p["right_rows"] if driving_is_right else p["left_rows"]
        detail = {"join": label, "on": on, "rows_before": rows_in, "rows_after": rows_out, **p}

        if p["max_right_per_key"] > 1:
            for a in j.left:
                repeated[a] = max(repeated.get(a, 1), p["max_right_per_key"])
        if p["max_left_per_key"] > 1:
            for a in j.right:
                repeated[a] = max(repeated.get(a, 1), p["max_left_per_key"])

        drv_repeat = p["max_left_per_key"] if driving_is_right else p["max_right_per_key"]
        if drv_repeat > 1:
            drv, other = (j.right, j.left) if driving_is_right else (j.left, j.right)
            fanned_out |= drv
            ratio = rows_out / rows_in if rows_in else float("inf")
            out.append(ReviewWarning(
                "JOIN_FANOUT", WARN,
                f"{label} ON {on}: row count goes {rows_in:,} → {rows_out:,} ({ratio:.2f}x). "
                f"A {self._label(drv)} row can match up to {drv_repeat} {self._label(other)} rows, "
                f"so {self._label(drv)} rows are repeated.", detail))
        if j.join_type == "INNER" and p["left_rows"] > p["left_matched"]:
            dropped = p["left_rows"] - p["left_matched"]
            out.append(ReviewWarning(
                "JOIN_DROPS_ROWS", WARN,
                f"{label} ON {on}: {dropped:,} of {p['left_rows']:,} {self._label(j.left)} rows have "
                f"no match (including NULL keys) and are removed by the INNER JOIN.",
                {**detail, "dropped": dropped}))
        return out

    def _profile(self, j: _Join, label: str) -> dict | ReviewWarning:
        def skip(why: str) -> ReviewWarning:
            return ReviewWarning("UNREVIEWABLE", INFO, f"{label}: {why}; row-count impact not checked.",
                                 {"join": label})
        if not j.keys:
            if j.node.get("condition") is not None:
                return skip("join condition is not a plain column equality")
            if len(j.left) != 1 or len(j.right) != 1:
                return skip("cross join over a multi-table input")
            L, R = self.rels[next(iter(j.left))], self.rels[next(iter(j.right))]
            if not (L.sql and R.sql):
                return skip("an input could not be profiled")
            try:
                lr, rr = self.con.execute(f"SELECT (SELECT COUNT(*) FROM {L.from_sql()} _r), "
                                          f"(SELECT COUNT(*) FROM {R.from_sql()} _r)").fetchone()
            except duckdb.Error as exc:
                return skip(f"profiling failed ({exc})")
            return {"left_rows": lr, "right_rows": rr, "matched_rows": lr * rr,
                    "left_matched": lr if rr else 0, "right_matched": rr if lr else 0,
                    "max_right_per_key": rr, "max_left_per_key": lr}

        l_aliases, r_aliases = {k[0] for k in j.keys}, {k[2] for k in j.keys}
        if len(l_aliases) != 1 or len(r_aliases) != 1:
            return skip("join keys span several tables on one side")
        L, R = self.rels[l_aliases.pop()], self.rels[r_aliases.pop()]
        if not (L.sql and R.sql):
            return skip("an input could not be profiled")

        def key_counts(rel: _Rel, cols: list[str]) -> str:
            sel = ", ".join(f"{_q(c)} AS _k{i}" for i, c in enumerate(cols))
            return f"SELECT {sel}, COUNT(*) AS _n FROM {rel.from_sql()} AS _r GROUP BY ALL"

        match = " AND ".join(f"l._k{i} = r._k{i}" for i in range(len(j.keys)))
        sql = f"""
            WITH l AS ({key_counts(L, [k[1] for k in j.keys])}),
                 r AS ({key_counts(R, [k[3] for k in j.keys])}),
                 m AS (SELECT l._n AS ln, r._n AS rn FROM l JOIN r ON {match})
            SELECT (SELECT COALESCE(SUM(_n), 0) FROM l)     AS left_rows,
                   (SELECT COALESCE(SUM(_n), 0) FROM r)     AS right_rows,
                   (SELECT COALESCE(SUM(ln * rn), 0) FROM m) AS matched_rows,
                   (SELECT COALESCE(SUM(ln), 0) FROM m)     AS left_matched,
                   (SELECT COALESCE(SUM(rn), 0) FROM m)     AS right_matched,
                   (SELECT COALESCE(MAX(rn), 0) FROM m)     AS max_right_per_key,
                   (SELECT COALESCE(MAX(ln), 0) FROM m)     AS max_left_per_key"""
        try:
            cur = self.con.execute(sql)
            names = [d[0] for d in cur.description]
            return {n: int(v) for n, v in zip(names, cur.fetchone())}
        except duckdb.Error as exc:
            return skip(f"profiling failed ({exc})")

    # ---- check 2: filters that drop NULLs ---------------------------------
    #
    # WHY THIS IS A CORRECTNESS ISSUE, NOT A STYLE ISSUE
    #
    # SQL predicates are three-valued: TRUE, FALSE, or UNKNOWN. Any comparison
    # with NULL (NULL = 'x', NULL <> 'x', NULL > 0, NULL IN (...)) is UNKNOWN,
    # and NOT UNKNOWN is still UNKNOWN. WHERE keeps only rows that are TRUE.
    #
    # So `status <> 'cancelled'` does not mean "everything except cancelled".
    # A row with NULL status passes neither `status = 'cancelled'` nor
    # `status <> 'cancelled'`: it disappears from a result AND from its
    # complement, so the two no longer sum to the total. That breaks the
    # reconciliation a board-level number depends on, and it changes the
    # number itself, not how the query reads. Nothing errors; the rows just
    # aren't there. Worse, a column with zero NULLs today will start dropping
    # rows the day an upstream load writes its first NULL.
    #
    # Two special cases make it sharper:
    #   * x NOT IN (1, 2, NULL) is never TRUE for any x, so the result is empty.
    #   * WHERE on a column from the optional side of a LEFT JOIN removes the
    #     unmatched rows (their column is NULL), silently turning it into an
    #     INNER JOIN.
    #
    # A predicate is treated as deliberate (not flagged) when the same column
    # also appears under IS [NOT] NULL, COALESCE/IFNULL, or IS [NOT] DISTINCT
    # FROM, because then the author has visibly decided what NULL means.
    def _check_where(self, outer_nullable: dict[str, dict]) -> list[ReviewWarning]:
        where = self.node.get("where_clause")
        if not where:
            return []
        guarded: set[tuple[str, str]] = set()
        for e in _walk_expr(where):
            if (_s(e.get("type")).upper() in _GUARD_TYPES
                    or _s(e.get("function_name")).lower() in _GUARD_FUNCS):
                for r in filter(None, map(self.resolve, _colrefs(e))):
                    guarded.add((r[0], r[1].lower()))

        out: list[ReviewWarning] = []
        seen: set = set()
        for pred, negated, op in _predicates(where):
            refs = [r for r in map(self.resolve, _colrefs(pred)) if r]
            if _is_column_equality(pred) and len({a for a, _ in refs}) > 1:
                continue                     # join condition written in WHERE
            if _not_in_with_null(pred, negated):
                out.append(ReviewWarning(
                    "NOT_IN_WITH_NULL", ERROR,
                    "NOT IN list contains NULL, so the predicate is never TRUE and the query returns no rows.",
                    {"operator": op}))
            for alias, col in refs:
                key = (alias, col.lower())
                if key in guarded or (key, op) in seen:
                    continue
                seen.add((key, op))
                rel = self.rels[alias]
                ref = f"{alias}.{col}"
                if alias in outer_nullable:
                    info = outer_nullable[alias]
                    lost = f" ({info['unmatched']:,} unmatched rows lost)" if info["unmatched"] else ""
                    out.append(ReviewWarning(
                        "LEFT_JOIN_NULLIFIED", ERROR,
                        f"WHERE `{ref} {op} …` filters the optional side of {info['join']}. Unmatched rows "
                        f"have NULL there, the predicate is UNKNOWN, and WHERE removes them, so the outer "
                        f"join now behaves as an INNER JOIN{lost}. Move the condition into ON, or add "
                        f"`OR {ref} IS NULL`.",
                        {"column": ref, "operator": op, "join": info["join"], "unmatched": info["unmatched"]}))
                    continue
                counts = self._null_count(rel, col)
                if counts is None:
                    continue
                nulls, total = counts
                detail = {"column": ref, "table": rel.label, "operator": op,
                          "null_rows": nulls, "total_rows": total, "negated": negated}
                if nulls > 0:
                    pct = nulls / total if total else 0.0
                    if negated:
                        msg = (f"`{ref} {op} …` silently excludes {nulls:,} of {total:,} {rel.label} rows "
                               f"({pct:.1%}) where {col} is NULL. Under three-valued logic those rows "
                               f"satisfy neither this filter nor its opposite, so this result and its "
                               f"complement won't add up to the total. Use `OR {ref} IS NULL` if unknowns "
                               f"belong in, or `IS DISTINCT FROM` for a null-safe comparison.")
                    else:
                        msg = (f"`{ref} {op} …` drops {nulls:,} of {total:,} {rel.label} rows ({pct:.1%}) "
                               f"where {col} is NULL. If excluding unknowns is intended, say so with "
                               f"`{ref} IS NOT NULL` so the exclusion is visible and testable.")
                    out.append(ReviewWarning("NULL_FILTER", ERROR if negated else INFO, msg, detail))
                elif self.include_latent and self.columns(rel).get(col.lower(), True):
                    out.append(ReviewWarning(
                        "NULL_FILTER_LATENT", INFO,
                        f"`{ref} {op} …` filters a nullable column with no NULLs today; it will start "
                        f"dropping rows silently if one arrives.", detail))
        return out

    def _null_count(self, rel: _Rel, col: str) -> tuple[int, int] | None:
        key = (rel.alias, col.lower())
        if key not in self._nulls:
            self._nulls[key] = None
            if rel.sql:
                try:
                    n, t = self.con.execute(
                        f"SELECT COUNT(*) - COUNT({_q(col)}), COUNT(*) FROM {rel.from_sql()} AS _r").fetchone()
                    self._nulls[key] = (int(n), int(t))
                except duckdb.Error:
                    pass
        return self._nulls[key]

    # ---- check 3: aggregates over repeated rows ----------------------------
    def _check_aggregates(self, repeated: dict[str, int], fanned_out: set[str]) -> list[ReviewWarning]:
        if not repeated:
            return []
        roots = list(self.node.get("select_list") or [])
        if self.node.get("having"):
            roots.append(self.node["having"])
        out: list[ReviewWarning] = []
        for root in roots:
            for e in _walk_expr(root):
                if (e.get("class") or "").upper() not in ("FUNCTION", "WINDOW"):
                    continue
                fn = (e.get("function_name") or "").lower()
                if fn not in _FANOUT_SENSITIVE_AGGS or e.get("distinct"):
                    continue
                refs = [r for r in map(self.resolve, _colrefs(e.get("children") or [])) if r]
                if fn == "count_star" or (fn == "count" and not refs):
                    if fanned_out:
                        out.append(ReviewWarning(
                            "AGG_OVER_FANOUT", WARN,
                            f"COUNT(*) counts rows of the joined result, and {self._label(fanned_out)} rows "
                            f"are repeated by a join, so it is not a count of {self._label(fanned_out)}. "
                            f"Use COUNT(DISTINCT <key>) if that is what you mean.",
                            {"function": "count_star", "repeated": sorted(fanned_out)}))
                    continue
                for alias, col in refs:
                    if alias in repeated:
                        rel = self.rels[alias]
                        out.append(ReviewWarning(
                            "AGG_OVER_FANOUT", ERROR,
                            f"{fn.upper()}({alias}.{col}) runs over {rel.label} rows that a join repeats up "
                            f"to {repeated[alias]}x per key, so the result is {_FANOUT_SENSITIVE_AGGS[fn]}. "
                            f"Aggregate {rel.label} before joining, or join to a relation that is unique "
                            f"on the key.",
                            {"function": fn, "column": f"{alias}.{col}", "table": rel.label,
                             "max_repeat": repeated[alias]}))
        return out

In [12]:
# Run review_query on your Phase 1 and Phase 2 queries.

tests = {
    "P1 joined rows (JOINED_SQL)": JOINED_SQL,
    "P1 plain rows (PLAIN_SQL)": PLAIN_SQL,
    "P1 joined average, as I ran it": f"SELECT AVG(revenue) FROM ({JOINED_SQL})",
    "P1 joined average, inline": """SELECT AVG(c.revenue) FROM customers c
                                    JOIN orders o ON o.customer_id = c.customer_id
                                    WHERE c.region = 'West'""",
    "P1 fixed query (CTE)": GOOD_SQL,
    "P2 != 'West'": "SELECT COUNT(*) FROM customers WHERE region != 'West'",
    "P2 IS DISTINCT FROM": "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'",
    "P2 window query": WINDOW_SQL,
}

for name, sql in tests.items():
    ws = review_query(con, sql)
    print(f"\n{name}: {len(ws)} warning(s)")
    for w in ws:
        print(f"  [{w.severity}] {w.code}: {w.message}")


P1 joined rows (JOINED_SQL): 2 warning(s)
  [warn] JOIN_FANOUT: customers INNER JOIN orders ON c.customer_id = o.customer_id: row count goes 3,000 → 20,584 (6.86x). A customers row can match up to 57 orders rows, so customers rows are repeated.
  [info] NULL_FILTER: `c.region = …` drops 450 of 3,000 customers rows (15.0%) where region is NULL. If excluding unknowns is intended, say so with `c.region IS NOT NULL` so the exclusion is visible and testable.

P1 plain rows (PLAIN_SQL): 1 warning(s)
  [info] NULL_FILTER: `customers.region = …` drops 450 of 3,000 customers rows (15.0%) where region is NULL. If excluding unknowns is intended, say so with `customers.region IS NOT NULL` so the exclusion is visible and testable.

P1 joined average, as I ran it: 2 warning(s)
  [warn] JOIN_FANOUT: customers INNER JOIN orders ON c.customer_id = o.customer_id: row count goes 3,000 → 20,584 (6.86x). A customers row can match up to 57 orders rows, so customers rows are repeated.
  [info] NULL_FILTER: 

*Your answer here.*

**What it caught:**

In Phase 1, `review_query` detected join fan-out, showing that the join increased rows from 3,000 to 20,584 and caused customers with more orders to receive more weight in the average. In Phase 2, it flagged `region != 'West'` for excluding NULL regions. It raised no warn- or error-level flags on the fixed queries, only an info note about the NULL regions.

**What it missed:**

1. **The actual board query:** Since the average was calculated in an outer query, the tool detected fan-out in the subquery but did not raise an `AGG_OVER_FANOUT` error.
2. **Accurate row counts:** It ignored `WHERE` conditions, reporting 3,000 to 20,584 instead of the actual 2,097 to 14,191 West customers and orders.
3. **False alarms:** It initially warned about `region = 'West'` even though excluding NULL values was intentional. I changed it to an info message.
4. **The [Evaluate] step:** The model could not access my queries or data, so it did not actually run the evaluation. Its tests used a small synthetic dataset with a different issue (a duplicated customer row), so they did not prove it could catch my problem. I tested the function on my own queries instead.

**Would it have caught the $1,240 before it reached the board?**

Partly. It detected the fan-out but only gave a warning because the average was in an outer query. Since correct queries also triggered warnings, the problem could easily have been overlooked.

### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*

**To:** Director  
**Re:** West Revenue and the Account-Management Programme

The correct average revenue for West customers is \$814, not $1,251. The higher number came from joining customers to orders, which counted customers multiple times based on their order volume. This inflated the average by about 54%. The row count revealed the issue: 2,097 customers became 14,191 rows.

Higher West revenue also does not prove that the programme works because West had larger accounts from the beginning. In my simulation, the true programme effect was \$25, but the simple comparison showed \$250. Controlling for account size brought the estimate back to \$25. However,  controlling for support tickets changed it to a \$29 loss because tickets are affected by both the programme and customer spending. We should only control for factors that existed before the programme started.

**My recommendations are:**

1. Check row counts after every join and aggregate data before joining when necessary.
2. Handle missing values explicitly, since 15% of customers have no recorded region and `region != 'West'` silently excludes them.
3. Use automated query reviews as a first check, not a final safeguard, since it only flagged the fan-out as a warning, not an error, when the average was in an outer query.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

**Model and date:** Claude Opus 5.5 / Nov 1

**Prompt:**


[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

**Reply:**

The reviewer is written, but I couldn’t do the Evaluate step, and I’m not going to guess at it. I don’t have your Phase 1 queries or the context behind the $1,240: nothing was uploaded and I have no memory of earlier chats. I also couldn’t run any code here. This sandbox has no network access, so I couldn’t install DuckDB. Both files pass a Python syntax check, but neither has been executed yet.

How it’s built

review_query(con, sql) in query_review.py returns a list of ReviewWarning(code, severity, message, detail) objects. They’re sorted errors first, and each code is stable, so your tests can assert on it.

It parses the query with DuckDB’s own json_serialize_sql(). That means the reviewer reads the query the same way the engine that runs it will. There’s no third-party parser, so it uses only duckdb and pandas.

It never runs your query. It only runs small profiling queries against customers and orders.

The three required checks

JOIN_FANOUT and JOIN_DROPS_ROWS. These flag a join that changes the row count, in either direction. Rows going missing in an INNER JOIN is the other common way a board number drifts, so it gets its own code.
NULL_FILTER. This flags a filter that drops rows where the column is NULL.
It’s an error when the filter is negated (<>, NOT IN, NOT LIKE), and a warn otherwise.
Two sharper cases have their own codes. LEFT_JOIN_NULLIFIED covers a WHERE condition on the optional side of a LEFT JOIN. NOT_IN_WITH_NULL covers NOT IN lists that contain NULL.
It stays quiet when you’ve visibly decided what NULL means, using IS [NOT] NULL, COALESCE, or IS DISTINCT FROM.
AGG_OVER_FANOUT. This is an error for SUM, AVG, COUNT and similar over rows a join has repeated. It’s a warn for COUNT(*) after fan-out. MIN and MAX are deliberately exempt, because repeating a row can’t change them.

The two mechanisms

Both are explained in the comments in the code. In short:

Fan-out. A join’s output size is set entirely by how often each key appears on each side: |L ⋈ R| = Σₖ n_L(k)·n_R(k). So the reviewer groups each side by its join key, joins those two small summaries, and reads off the exact row count and the largest number of matches per key. It never builds the join’s output, which is exactly the thing that explodes when there’s fan-out.
Why NULL filters are a correctness issue. A comparison with NULL is UNKNOWN, not FALSE, and NOT UNKNOWN is still UNKNOWN. WHERE keeps only rows that are TRUE. So a row with a NULL status fails both status = 'cancelled' and status <> 'cancelled'. It vanishes from the result and from its complement, so the two no longer add up to the total. That changes the number itself, and nothing errors.

Tests

test_query_review.py builds a small customers/orders dataset with three planted problems:

a customer inserted twice after an address change
an order with a NULL status
an order whose customer is missing

It also checks that a correctly written query raises no warnings.

What to check on your first run

My knowledge of DuckDB’s AST key names (COMPARE_NOTEQUAL, count_star, join_type, and so on) comes from memory, not from running it. They’re all collected in one helper section of the code. If a test fails, that section is the first place to look.

There’s also a known limitation. Key profiles cover the whole base table, not the rows left after your WHERE filters. That means it can over-report duplication, but it shouldn’t miss any.

Next step

Paste both Phase 1 queries into PHASE1_QUERIES and run pytest -s, or paste them here and tell me where the $1,240 showed up. Then I can tell you which check catches it, or whether it falls outside all three, which would be worth knowing too.

```
"""
query_review.py: pre-dashboard SQL review for DuckDB.

    warnings = review_query(con, sql)
    warnings_to_frame(warnings)   # optional: a pandas DataFrame for eyeballing

Dependencies: duckdb and pandas only. The SQL is parsed by DuckDB itself via
json_serialize_sql(), so there is no third-party parser and no chance that
the reviewer reads the query differently from the engine that will run it.

Checks (warning codes are stable so tests can assert on them):
    JOIN_FANOUT          a join repeats rows of the driving table (row count grows)
    JOIN_DROPS_ROWS      an INNER JOIN removes unmatched rows (row count shrinks)
    NULL_FILTER          a WHERE predicate silently drops rows whose column is NULL
    LEFT_JOIN_NULLIFIED  a WHERE predicate on the optional side of an outer join
    NOT_IN_WITH_NULL     NOT IN (..., NULL, ...) can never be TRUE
    AGG_OVER_FANOUT      SUM/AVG/COUNT/... over rows that a join has repeated
    NULL_FILTER_LATENT   (opt-in) filter on a nullable column that has no NULLs today
    UNREVIEWABLE         something we could not analyse; never silently skipped
    UNPARSEABLE          DuckDB could not parse the statement as a SELECT
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from typing import Any, Iterator

import duckdb
import pandas as pd

ERROR, WARN, INFO = "error", "warn", "info"
_SEVERITY_RANK = {ERROR: 0, WARN: 1, INFO: 2}


@dataclass
class ReviewWarning:
    code: str
    severity: str
    message: str
    detail: dict[str, Any] = field(default_factory=dict)


def warnings_to_frame(warnings: list[ReviewWarning]) -> pd.DataFrame:
    return pd.DataFrame([asdict(w) for w in warnings],
                        columns=["code", "severity", "message", "detail"])


# --------------------------------------------------------------------------
# Public entry point
# --------------------------------------------------------------------------
def review_query(con: duckdb.DuckDBPyConnection, sql: str, *,
                 include_latent_nulls: bool = False) -> list[ReviewWarning]:
    """Return structured warnings for a single SELECT. Never executes `sql`
    itself; only small profiling queries against the tables it reads."""
    try:
        raw = con.execute("SELECT json_serialize_sql(?)", [sql]).fetchone()[0]
        tree = json.loads(raw)
    except duckdb.Error as exc:
        return [ReviewWarning("UNPARSEABLE", ERROR, f"DuckDB could not parse the query: {exc}")]
    if tree.get("error"):
        return [ReviewWarning("UNPARSEABLE", ERROR,
                              f"DuckDB could not parse the query as a SELECT: "
                              f"{tree.get('error_message', 'unknown error')}")]
    statements = tree.get("statements", [])

    # CTE name -> its SELECT statement, so `FROM my_cte` can be profiled.
    ctes: dict[str, dict] = {}
    for node in _walk(statements):
        for entry in (node.get("cte_map") or {}).get("map", []) or []:
            query = (entry.get("value") or {}).get("query")
            if entry.get("key") and query:
                ctes[entry["key"].lower()] = query

    # Every SELECT_NODE (outer query, CTE bodies, subqueries) is its own scope.
    warnings: list[ReviewWarning] = []
    for node in _walk(statements):
        if node.get("type") == "SELECT_NODE":
            warnings += _Scope(con, node, ctes, include_latent_nulls).review()

    unique, seen = [], set()
    for w in warnings:
        if (w.code, w.message) not in seen:
            seen.add((w.code, w.message))
            unique.append(w)
    return sorted(unique, key=lambda w: _SEVERITY_RANK[w.severity])


# --------------------------------------------------------------------------
# AST helpers (all knowledge of DuckDB's JSON AST shape lives in this file)
# --------------------------------------------------------------------------
_OP_SYMBOL = {
    "COMPARE_EQUAL": "=", "COMPARE_NOTEQUAL": "<>",
    "COMPARE_LESSTHAN": "<", "COMPARE_GREATERTHAN": ">",
    "COMPARE_LESSTHANOREQUALTO": "<=", "COMPARE_GREATERTHANOREQUALTO": ">=",
    "COMPARE_IN": "IN", "COMPARE_NOT_IN": "NOT IN",
    "COMPARE_BETWEEN": "BETWEEN", "COMPARE_NOT_BETWEEN": "NOT BETWEEN",
}
_NEGATING_OPS = {"COMPARE_NOTEQUAL", "COMPARE_NOT_IN", "COMPARE_NOT_BETWEEN"}
_LIKE_FUNCS = {"~~": "LIKE", "!~~": "NOT LIKE", "~~*": "ILIKE", "!~~*": "NOT ILIKE",
               "like": "LIKE", "not_like": "NOT LIKE", "ilike": "ILIKE",
               "not_ilike": "NOT ILIKE", "regexp_matches": "regexp_matches",
               "contains": "contains", "starts_with": "starts_with",
               "prefix": "prefix", "suffix": "suffix"}
_GUARD_TYPES = {"OPERATOR_IS_NULL", "OPERATOR_IS_NOT_NULL", "OPERATOR_COALESCE",
                "COMPARE_DISTINCT_FROM", "COMPARE_NOT_DISTINCT_FROM"}
_GUARD_FUNCS = {"coalesce", "ifnull", "nvl"}

# Aggregates whose result changes when input rows are duplicated. MIN, MAX,
# BOOL_AND/OR, ANY_VALUE are idempotent (f(x, x) = f(x)) so are left out.
_FANOUT_SENSITIVE_AGGS = {
    "sum": "inflated", "fsum": "inflated", "sumkahan": "inflated",
    "kahan_sum": "inflated", "product": "inflated",
    "count": "inflated", "count_star": "inflated",
    "avg": "weighted toward keys with more matches",
    "mean": "weighted toward keys with more matches",
    "median": "weighted toward keys with more matches",
    "quantile_cont": "weighted toward keys with more matches",
    "stddev_samp": "distorted", "stddev_pop": "distorted",
    "string_agg": "padded with repeats", "list": "padded with repeats",
    "array_agg": "padded with repeats",
}


def _walk(node: Any) -> Iterator[dict]:
    """Every dict in the tree, including nested queries."""
    if isinstance(node, dict):
        yield node
        for v in node.values():
            yield from _walk(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk(v)


def _walk_expr(node: Any) -> Iterator[dict]:
    """Every expression dict, without descending into subqueries; those have
    their own scope and are reviewed as their own SELECT_NODE."""
    if isinstance(node, dict):
        yield node
        if (node.get("class") or "").upper() == "SUBQUERY":
            yield from _walk_expr(node.get("child"))
            return
        for v in node.values():
            yield from _walk_expr(v)
    elif isinstance(node, list):
        for v in node:
            yield from _walk_expr(v)


def _colrefs(expr: Any) -> list[dict]:
    return [e for e in _walk_expr(expr) if (e.get("class") or "").upper() == "COLUMN_REF"]


def _conjuncts(expr: dict | None) -> list[dict]:
    if not expr:
        return []
    if (expr.get("type") or "").upper() == "CONJUNCTION_AND":
        return [x for c in expr.get("children", []) for x in _conjuncts(c)]
    return [expr]


def _predicates(e: Any, negated: bool = False) -> Iterator[tuple[dict, bool, str]]:
    """Yield (predicate, effectively_negated, display_operator) for every
    null-sensitive leaf predicate, tracking NOT through AND/OR."""
    if not isinstance(e, dict):
        return
    t, cls = (e.get("type") or "").upper(), (e.get("class") or "").upper()
    if t == "OPERATOR_NOT":
        for c in e.get("children", []):
            yield from _predicates(c, not negated)
    elif t in ("CONJUNCTION_AND", "CONJUNCTION_OR"):
        for c in e.get("children", []):
            yield from _predicates(c, negated)
    elif cls == "SUBQUERY":
        if (e.get("subquery_type") or "").upper() == "ANY" and e.get("child"):
            yield e, negated, "NOT IN (subquery)" if negated else "IN (subquery)"
    elif t in _OP_SYMBOL:
        sym = _OP_SYMBOL[t]
        yield e, negated ^ (t in _NEGATING_OPS), f"NOT {sym}" if negated else sym
    elif cls == "FUNCTION" and (e.get("function_name") or "").lower() in _LIKE_FUNCS:
        name = _LIKE_FUNCS[e["function_name"].lower()]
        yield e, negated ^ name.startswith("NOT"), f"NOT {name}" if negated else name


def _is_column_equality(e: dict) -> bool:
    return ((e.get("type") or "").upper() == "COMPARE_EQUAL"
            and (e.get("left") or {}).get("class") == "COLUMN_REF"
            and (e.get("right") or {}).get("class") == "COLUMN_REF")


def _not_in_with_null(pred: dict, negated: bool) -> bool:
    if not negated or (pred.get("type") or "").upper() not in ("COMPARE_IN", "COMPARE_NOT_IN"):
        return False
    return any((c.get("class") or "").upper() == "CONSTANT" and (c.get("value") or {}).get("is_null")
               for c in (pred.get("children") or [])[1:])


def _q(ident: str) -> str:
    return '"' + ident.replace('"', '""') + '"'


# --------------------------------------------------------------------------
# One SELECT scope
# --------------------------------------------------------------------------
@dataclass
class _Rel:
    alias: str
    label: str
    sql: str | None            # SQL producing this relation's rows; None = cannot profile
    is_table: bool
    columns: dict[str, bool] | None = None   # lower(name) -> declared nullable

    def from_sql(self) -> str:
        return self.sql if self.is_table else f"({self.sql})"


@dataclass
class _Join:
    join_type: str
    ref_type: str
    left: set[str]
    right: set[str]
    node: dict
    keys: list[tuple[str, str, str, str]] = field(default_factory=list)  # (l_alias, l_col, r_alias, r_col)


class _Scope:
    def __init__(self, con, node: dict, ctes: dict[str, dict], include_latent: bool):
        self.con, self.node, self.ctes, self.include_latent = con, node, ctes, include_latent
        self.rels: dict[str, _Rel] = {}
        self.joins: list[_Join] = []
        self._nulls: dict[tuple[str, str], tuple[int, int] | None] = {}
        if node.get("from_table"):
            self._flatten(node["from_table"])
        # Comma joins (FROM a, b WHERE a.id = b.id) parse as CROSS joins with
        # the key in WHERE; recover those keys so they aren't mistaken for
        # a cartesian product.
        for j in self.joins:
            if not j.keys and j.node.get("condition") is None:
                for conj in _conjuncts(node.get("where_clause")):
                    k = self._equi_pair(conj, j.left, j.right)
                    if k:
                        j.keys.append(k)

    # ---- relations and column resolution --------------------------------
    def _flatten(self, t: dict) -> set[str]:
        kind = (t.get("type") or "").upper()
        if kind == "JOIN":
            left, right = self._flatten(t["left"]), self._flatten(t["right"])
            j = _Join((t.get("join_type") or "INNER").upper(),
                      (t.get("ref_type") or "REGULAR").upper(), left, right, t)
            for c in t.get("using_columns") or []:
                la, ra = self._owner(c, left), self._owner(c, right)
                if la and ra:
                    j.keys.append((la, c, ra, c))
            for conj in _conjuncts(t.get("condition")):
                k = self._equi_pair(conj, left, right)
                if k:
                    j.keys.append(k)
            self.joins.append(j)
            return left | right
        alias = (t.get("alias") or t.get("table_name") or f"_rel{len(self.rels)}").lower()
        self.rels[alias] = self._make_rel(alias, t, kind)
        return {alias}

    def _make_rel(self, alias: str, t: dict, kind: str) -> _Rel:
        if kind == "BASE_TABLE":
            name = t["table_name"]
            if name.lower() in self.ctes:
                return _Rel(alias, f"CTE {name}", self._to_sql(self.ctes[name.lower()]), False)
            parts = [p for p in (t.get("catalog_name"), t.get("schema_name"), name) if p]
            return _Rel(alias, name, ".".join(_q(p) for p in parts), True)
        if kind == "SUBQUERY":
            return _Rel(alias, f"subquery {alias}", self._to_sql(t.get("subquery")), False)
        return _Rel(alias, f"{kind.lower()} {alias}", None, False)

    def _to_sql(self, stmt: dict | None) -> str | None:
        """Turn a parsed sub-statement back into SQL with DuckDB's own
        json_deserialize_sql, so subqueries and CTEs can be profiled."""
        if not stmt:
            return None
        try:
            payload = json.dumps({"error": False, "statements": [stmt]})
            return self.con.execute("SELECT json_deserialize_sql(?::JSON)", [payload]).fetchone()[0]
        except duckdb.Error:
            return None

    def columns(self, rel: _Rel) -> dict[str, bool]:
        if rel.columns is None:
            rel.columns = {}
            if rel.sql:
                try:
                    df = self.con.execute(f"DESCRIBE {rel.sql}").df()
                    rel.columns = {str(n).lower(): str(nl).upper() != "NO"
                                   for n, nl in zip(df["column_name"], df["null"])}
                except duckdb.Error:
                    pass
        return rel.columns

    def _owner(self, col: str, aliases: set[str]) -> str | None:
        owners = [a for a in aliases if col.lower() in self.columns(self.rels[a])]
        return owners[0] if len(owners) == 1 else None

    def resolve(self, colref: dict) -> tuple[str, str] | None:
        names = colref.get("column_names") or []
        if not names:
            return None
        col = names[-1]
        if len(names) >= 2:
            q = names[-2].lower()
            if q in self.rels:
                return q, col
            for a, r in self.rels.items():          # qualified by table name, not alias
                if r.label.lower() == q:
                    return a, col
            return None                             # correlated / outer reference
        owner = self._owner(col, set(self.rels))
        return (owner, col) if owner else None

    def _equi_pair(self, e: dict, left: set[str], right: set[str]):
        if not _is_column_equality(e):
            return None
        a, b = self.resolve(e["left"]), self.resolve(e["right"])
        if not a or not b:
            return None
        if a[0] in left and b[0] in right:
            return (*a, *b)
        if b[0] in left and a[0] in right:
            return (*b, *a)
        return None

    def _label(self, aliases: set[str]) -> str:
        return ", ".join(self.rels[a].label for a in sorted(aliases))

    # ---- the review ----------------------------------------------------
    def review(self) -> list[ReviewWarning]:
        warnings: list[ReviewWarning] = []
        repeated: dict[str, int] = {}       # alias -> max times a row is repeated by some join
        fanned_out: set[str] = set()        # aliases repeated beyond the driving table's grain
        outer_nullable: dict[str, dict] = {}
        for j in self.joins:
            warnings += self._check_join(j, repeated, fanned_out, outer_nullable)
        warnings += self._check_where(outer_nullable)
        warnings += self._check_aggregates(repeated, fanned_out)
        return warnings

    # ---- check 1: joins that change the row count -------------------------
    #
    # HOW FAN-OUT IS DETECTED WITHOUT RUNNING THE JOIN (TWICE OR AT ALL)
    #
    # The naive test is COUNT(*) of the query with and without the join,
    # which runs the expensive join and then runs it again for every variant.
    # Instead we use the fact that an equi-join's output size is fully
    # determined by key multiplicities:
    #
    #     |L ⋈ R| = Σ over keys k of  n_L(k) · n_R(k)
    #
    # where n_L(k) is how many rows of L carry key k. So we GROUP BY the join
    # key on each side separately (one scan per table, output size = number
    # of distinct keys), join those two small summaries, and read off:
    #
    #     matched_rows       Σ n_L·n_R                  exact inner-join row count
    #     left_matched       Σ n_L over matched keys    L rows that find a partner
    #     max_right_per_key  max n_R                    >1  ⇒ L rows get repeated
    #     max_left_per_key   max n_L                    >1  ⇒ R rows get repeated
    #
    # LEFT/RIGHT/FULL counts follow by adding back unmatched rows. NULL keys
    # fall out naturally, because the summary join uses `=`, exactly as the
    # real join does. The cost is two GROUP BYs, never the join's output
    # (which is the thing that explodes when there is fan-out).
    #
    # Approximation: for a chain of joins we profile against the base table
    # that owns each key, and over the whole table rather than after the
    # query's WHERE. That can over-report, never under-report, duplication.
    def _check_join(self, j: _Join, repeated, fanned_out, outer_nullable) -> list[ReviewWarning]:
        if j.ref_type in ("ASOF", "POSITIONAL") or j.join_type in ("SEMI", "ANTI"):
            return []      # these joins cannot repeat rows
        label = f"{self._label(j.left)} {j.join_type} JOIN {self._label(j.right)}"
        on = " AND ".join(f"{la}.{lc} = {ra}.{rc}" for la, lc, ra, rc in j.keys) or "(no equi-key)"
        p = self._profile(j, label)
        out: list[ReviewWarning] = []
        if isinstance(p, ReviewWarning):
            out.append(p)
            p = None

        if j.join_type in ("LEFT", "OUTER"):
            for a in j.right:
                outer_nullable[a] = {"join": label,
                                     "unmatched": p and p["left_rows"] - p["left_matched"]}
        if j.join_type in ("RIGHT", "OUTER"):
            for a in j.left:
                outer_nullable[a] = {"join": label,
                                     "unmatched": p and p["right_rows"] - p["right_matched"]}
        if p is None:
            return out

        rows_out = p["matched_rows"]
        if j.join_type in ("LEFT", "OUTER"):
            rows_out += p["left_rows"] - p["left_matched"]
        if j.join_type in ("RIGHT", "OUTER"):
            rows_out += p["right_rows"] - p["right_matched"]
        driving_is_right = j.join_type == "RIGHT"
        rows_in = p["right_rows"] if driving_is_right else p["left_rows"]
        detail = {"join": label, "on": on, "rows_before": rows_in, "rows_after": rows_out, **p}

        if p["max_right_per_key"] > 1:
            for a in j.left:
                repeated[a] = max(repeated.get(a, 1), p["max_right_per_key"])
        if p["max_left_per_key"] > 1:
            for a in j.right:
                repeated[a] = max(repeated.get(a, 1), p["max_left_per_key"])

        drv_repeat = p["max_left_per_key"] if driving_is_right else p["max_right_per_key"]
        if drv_repeat > 1:
            drv, other = (j.right, j.left) if driving_is_right else (j.left, j.right)
            fanned_out |= drv
            ratio = rows_out / rows_in if rows_in else float("inf")
            out.append(ReviewWarning(
                "JOIN_FANOUT", WARN,
                f"{label} ON {on}: row count goes {rows_in:,} → {rows_out:,} ({ratio:.2f}x). "
                f"A {self._label(drv)} row can match up to {drv_repeat} {self._label(other)} rows, "
                f"so {self._label(drv)} rows are repeated.", detail))
        if j.join_type == "INNER" and p["left_rows"] > p["left_matched"]:
            dropped = p["left_rows"] - p["left_matched"]
            out.append(ReviewWarning(
                "JOIN_DROPS_ROWS", WARN,
                f"{label} ON {on}: {dropped:,} of {p['left_rows']:,} {self._label(j.left)} rows have "
                f"no match (including NULL keys) and are removed by the INNER JOIN.",
                {**detail, "dropped": dropped}))
        return out

    def _profile(self, j: _Join, label: str) -> dict | ReviewWarning:
        def skip(why: str) -> ReviewWarning:
            return ReviewWarning("UNREVIEWABLE", INFO, f"{label}: {why}; row-count impact not checked.",
                                 {"join": label})
        if not j.keys:
            if j.node.get("condition") is not None:
                return skip("join condition is not a plain column equality")
            if len(j.left) != 1 or len(j.right) != 1:
                return skip("cross join over a multi-table input")
            L, R = self.rels[next(iter(j.left))], self.rels[next(iter(j.right))]
            if not (L.sql and R.sql):
                return skip("an input could not be profiled")
            try:
                lr, rr = self.con.execute(f"SELECT (SELECT COUNT(*) FROM {L.from_sql()} _r), "
                                          f"(SELECT COUNT(*) FROM {R.from_sql()} _r)").fetchone()
            except duckdb.Error as exc:
                return skip(f"profiling failed ({exc})")
            return {"left_rows": lr, "right_rows": rr, "matched_rows": lr * rr,
                    "left_matched": lr if rr else 0, "right_matched": rr if lr else 0,
                    "max_right_per_key": rr, "max_left_per_key": lr}

        l_aliases, r_aliases = {k[0] for k in j.keys}, {k[2] for k in j.keys}
        if len(l_aliases) != 1 or len(r_aliases) != 1:
            return skip("join keys span several tables on one side")
        L, R = self.rels[l_aliases.pop()], self.rels[r_aliases.pop()]
        if not (L.sql and R.sql):
            return skip("an input could not be profiled")

        def key_counts(rel: _Rel, cols: list[str]) -> str:
            sel = ", ".join(f"{_q(c)} AS _k{i}" for i, c in enumerate(cols))
            return f"SELECT {sel}, COUNT(*) AS _n FROM {rel.from_sql()} AS _r GROUP BY ALL"

        match = " AND ".join(f"l._k{i} = r._k{i}" for i in range(len(j.keys)))
        sql = f"""
            WITH l AS ({key_counts(L, [k[1] for k in j.keys])}),
                 r AS ({key_counts(R, [k[3] for k in j.keys])}),
                 m AS (SELECT l._n AS ln, r._n AS rn FROM l JOIN r ON {match})
            SELECT (SELECT COALESCE(SUM(_n), 0) FROM l)     AS left_rows,
                   (SELECT COALESCE(SUM(_n), 0) FROM r)     AS right_rows,
                   (SELECT COALESCE(SUM(ln * rn), 0) FROM m) AS matched_rows,
                   (SELECT COALESCE(SUM(ln), 0) FROM m)     AS left_matched,
                   (SELECT COALESCE(SUM(rn), 0) FROM m)     AS right_matched,
                   (SELECT COALESCE(MAX(rn), 0) FROM m)     AS max_right_per_key,
                   (SELECT COALESCE(MAX(ln), 0) FROM m)     AS max_left_per_key"""
        try:
            cur = self.con.execute(sql)
            names = [d[0] for d in cur.description]
            return {n: int(v) for n, v in zip(names, cur.fetchone())}
        except duckdb.Error as exc:
            return skip(f"profiling failed ({exc})")

    # ---- check 2: filters that drop NULLs ---------------------------------
    #
    # WHY THIS IS A CORRECTNESS ISSUE, NOT A STYLE ISSUE
    #
    # SQL predicates are three-valued: TRUE, FALSE, or UNKNOWN. Any comparison
    # with NULL (NULL = 'x', NULL <> 'x', NULL > 0, NULL IN (...)) is UNKNOWN,
    # and NOT UNKNOWN is still UNKNOWN. WHERE keeps only rows that are TRUE.
    #
    # So `status <> 'cancelled'` does not mean "everything except cancelled".
    # A row with NULL status passes neither `status = 'cancelled'` nor
    # `status <> 'cancelled'`: it disappears from a result AND from its
    # complement, so the two no longer sum to the total. That breaks the
    # reconciliation a board-level number depends on, and it changes the
    # number itself, not how the query reads. Nothing errors; the rows just
    # aren't there. Worse, a column with zero NULLs today will start dropping
    # rows the day an upstream load writes its first NULL.
    #
    # Two special cases make it sharper:
    #   * x NOT IN (1, 2, NULL) is never TRUE for any x, so the result is empty.
    #   * WHERE on a column from the optional side of a LEFT JOIN removes the
    #     unmatched rows (their column is NULL), silently turning it into an
    #     INNER JOIN.
    #
    # A predicate is treated as deliberate (not flagged) when the same column
    # also appears under IS [NOT] NULL, COALESCE/IFNULL, or IS [NOT] DISTINCT
    # FROM, because then the author has visibly decided what NULL means.
    def _check_where(self, outer_nullable: dict[str, dict]) -> list[ReviewWarning]:
        where = self.node.get("where_clause")
        if not where:
            return []
        guarded: set[tuple[str, str]] = set()
        for e in _walk_expr(where):
            if ((e.get("type") or "").upper() in _GUARD_TYPES
                    or (e.get("function_name") or "").lower() in _GUARD_FUNCS):
                for r in filter(None, map(self.resolve, _colrefs(e))):
                    guarded.add((r[0], r[1].lower()))

        out: list[ReviewWarning] = []
        seen: set = set()
        for pred, negated, op in _predicates(where):
            refs = [r for r in map(self.resolve, _colrefs(pred)) if r]
            if _is_column_equality(pred) and len({a for a, _ in refs}) > 1:
                continue                     # join condition written in WHERE
            if _not_in_with_null(pred, negated):
                out.append(ReviewWarning(
                    "NOT_IN_WITH_NULL", ERROR,
                    "NOT IN list contains NULL, so the predicate is never TRUE and the query returns no rows.",
                    {"operator": op}))
            for alias, col in refs:
                key = (alias, col.lower())
                if key in guarded or (key, op) in seen:
                    continue
                seen.add((key, op))
                rel = self.rels[alias]
                ref = f"{alias}.{col}"
                if alias in outer_nullable:
                    info = outer_nullable[alias]
                    lost = f" ({info['unmatched']:,} unmatched rows lost)" if info["unmatched"] else ""
                    out.append(ReviewWarning(
                        "LEFT_JOIN_NULLIFIED", ERROR,
                        f"WHERE `{ref} {op} …` filters the optional side of {info['join']}. Unmatched rows "
                        f"have NULL there, the predicate is UNKNOWN, and WHERE removes them, so the outer "
                        f"join now behaves as an INNER JOIN{lost}. Move the condition into ON, or add "
                        f"`OR {ref} IS NULL`.",
                        {"column": ref, "operator": op, "join": info["join"], "unmatched": info["unmatched"]}))
                    continue
                counts = self._null_count(rel, col)
                if counts is None:
                    continue
                nulls, total = counts
                detail = {"column": ref, "table": rel.label, "operator": op,
                          "null_rows": nulls, "total_rows": total, "negated": negated}
                if nulls > 0:
                    pct = nulls / total if total else 0.0
                    if negated:
                        msg = (f"`{ref} {op} …` silently excludes {nulls:,} of {total:,} {rel.label} rows "
                               f"({pct:.1%}) where {col} is NULL. Under three-valued logic those rows "
                               f"satisfy neither this filter nor its opposite, so this result and its "
                               f"complement won't add up to the total. Use `OR {ref} IS NULL` if unknowns "
                               f"belong in, or `IS DISTINCT FROM` for a null-safe comparison.")
                    else:
                        msg = (f"`{ref} {op} …` drops {nulls:,} of {total:,} {rel.label} rows ({pct:.1%}) "
                               f"where {col} is NULL. If excluding unknowns is intended, say so with "
                               f"`{ref} IS NOT NULL` so the exclusion is visible and testable.")
                    out.append(ReviewWarning("NULL_FILTER", ERROR if negated else WARN, msg, detail))
                elif self.include_latent and self.columns(rel).get(col.lower(), True):
                    out.append(ReviewWarning(
                        "NULL_FILTER_LATENT", INFO,
                        f"`{ref} {op} …` filters a nullable column with no NULLs today; it will start "
                        f"dropping rows silently if one arrives.", detail))
        return out

    def _null_count(self, rel: _Rel, col: str) -> tuple[int, int] | None:
        key = (rel.alias, col.lower())
        if key not in self._nulls:
            self._nulls[key] = None
            if rel.sql:
                try:
                    n, t = self.con.execute(
                        f"SELECT COUNT(*) - COUNT({_q(col)}), COUNT(*) FROM {rel.from_sql()} AS _r").fetchone()
                    self._nulls[key] = (int(n), int(t))
                except duckdb.Error:
                    pass
        return self._nulls[key]

    # ---- check 3: aggregates over repeated rows ----------------------------
    def _check_aggregates(self, repeated: dict[str, int], fanned_out: set[str]) -> list[ReviewWarning]:
        if not repeated:
            return []
        roots = list(self.node.get("select_list") or [])
        if self.node.get("having"):
            roots.append(self.node["having"])
        out: list[ReviewWarning] = []
        for root in roots:
            for e in _walk_expr(root):
                if (e.get("class") or "").upper() not in ("FUNCTION", "WINDOW"):
                    continue
                fn = (e.get("function_name") or "").lower()
                if fn not in _FANOUT_SENSITIVE_AGGS or e.get("distinct"):
                    continue
                refs = [r for r in map(self.resolve, _colrefs(e.get("children") or [])) if r]
                if fn == "count_star" or (fn == "count" and not refs):
                    if fanned_out:
                        out.append(ReviewWarning(
                            "AGG_OVER_FANOUT", WARN,
                            f"COUNT(*) counts rows of the joined result, and {self._label(fanned_out)} rows "
                            f"are repeated by a join, so it is not a count of {self._label(fanned_out)}. "
                            f"Use COUNT(DISTINCT <key>) if that is what you mean.",
                            {"function": "count_star", "repeated": sorted(fanned_out)}))
                    continue
                for alias, col in refs:
                    if alias in repeated:
                        rel = self.rels[alias]
                        out.append(ReviewWarning(
                            "AGG_OVER_FANOUT", ERROR,
                            f"{fn.upper()}({alias}.{col}) runs over {rel.label} rows that a join repeats up "
                            f"to {repeated[alias]}x per key, so the result is {_FANOUT_SENSITIVE_AGGS[fn]}. "
                            f"Aggregate {rel.label} before joining, or join to a relation that is unique "
                            f"on the key.",
                            {"function": fn, "column": f"{alias}.{col}", "table": rel.label,
                             "max_repeat": repeated[alias]}))
        return out
  
  """
Tests for review_query on synthetic customers/orders data.
Run:  pytest test_query_review.py -v      or     python test_query_review.py
"""
import duckdb
import pytest

from query_review import review_query, warnings_to_frame

# Paste the two Phase 1 queries here to run them through the reviewer.
PHASE1_QUERIES: dict[str, str] = {
    # "phase1_q1": """ ... """,
    # "phase1_q2": """ ... """,
}


@pytest.fixture
def con():
    c = duckdb.connect()
    c.execute("""
        CREATE TABLE customers (customer_id INTEGER, region VARCHAR, updated_at DATE);
        INSERT INTO customers VALUES
            (1, 'East', '2026-01-01'),
            (2, 'West', '2026-01-01'),
            (3, 'East', '2026-01-01'),
            (3, 'West', '2026-03-01');   -- customer 3 re-inserted on an address change

        CREATE TABLE orders (order_id INTEGER, customer_id INTEGER,
                             amount DECIMAL(10,2), status VARCHAR);
        INSERT INTO orders VALUES
            (10, 1, 100.00, 'paid'),
            (11, 2, 250.00, 'paid'),
            (12, 3, 400.00, 'paid'),
            (13, 3,  90.00, NULL),       -- status not yet loaded
            (14, 1,  60.00, 'cancelled'),
            (15, 4,  30.00, 'paid');     -- customer 4 missing from customers
    """)
    return c


def codes(ws):
    return {w.code for w in ws}


def test_duplicate_dimension_inflates_sum(con):
    ws = review_query(con, """
        SELECT c.region, SUM(o.amount) AS revenue
        FROM orders o JOIN customers c ON o.customer_id = c.customer_id
        GROUP BY c.region""")
    assert {"JOIN_FANOUT", "AGG_OVER_FANOUT", "JOIN_DROPS_ROWS"} <= codes(ws)
    agg = next(w for w in ws if w.code == "AGG_OVER_FANOUT")
    assert agg.severity == "error" and agg.detail["column"] == "o.amount"
    fan = next(w for w in ws if w.code == "JOIN_FANOUT")
    # 6 orders: 12 and 13 each match customer 3 twice (+2), order 15 matches nothing (−1)
    assert (fan.detail["rows_before"], fan.detail["rows_after"]) == (6, 7)


def test_not_equal_on_nullable_status(con):
    ws = review_query(con, "SELECT SUM(amount) FROM orders WHERE status <> 'cancelled'")
    nf = [w for w in ws if w.code == "NULL_FILTER"]
    assert len(nf) == 1 and nf[0].severity == "error" and nf[0].detail["null_rows"] == 1


def test_explicit_null_handling_is_not_flagged(con):
    ws = review_query(con, "SELECT SUM(amount) FROM orders WHERE status IS DISTINCT FROM 'cancelled'")
    assert "NULL_FILTER" not in codes(ws)


def test_where_on_left_join_optional_side(con):
    ws = review_query(con, """
        SELECT SUM(o.amount) FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.region = 'East'""")
    assert "LEFT_JOIN_NULLIFIED" in codes(ws)


def test_not_in_with_null(con):
    ws = review_query(con, "SELECT * FROM orders WHERE status NOT IN ('cancelled', NULL)")
    assert "NOT_IN_WITH_NULL" in codes(ws)


def test_count_star_after_fanout(con):
    ws = review_query(con, """
        SELECT COUNT(*) FROM orders o JOIN customers c ON o.customer_id = c.customer_id""")
    assert any(w.code == "AGG_OVER_FANOUT" and w.detail["function"] == "count_star" for w in ws)


def test_clean_query_has_no_errors_or_warnings(con):
    ws = review_query(con, """
        WITH latest AS (
            SELECT DISTINCT ON (customer_id) customer_id, region
            FROM customers ORDER BY customer_id, updated_at DESC)
        SELECT l.region, SUM(o.amount) AS revenue
        FROM orders o LEFT JOIN latest l ON o.customer_id = l.customer_id
        WHERE o.status IS DISTINCT FROM 'cancelled'
        GROUP BY l.region""")
    assert [w for w in ws if w.severity in ("error", "warn")] == []


def test_returns_structured_warning_on_bad_sql(con):
    ws = review_query(con, "SELEKT 1")
    assert codes(ws) == {"UNPARSEABLE"}


@pytest.mark.parametrize("name", sorted(PHASE1_QUERIES))
def test_phase1(con, name):
    ws = review_query(con, PHASE1_QUERIES[name])
    print(f"\n{name}\n", warnings_to_frame(ws)[["code", "severity", "message"]].to_string())


if __name__ == "__main__":
    raise SystemExit(pytest.main([__file__, "-v", "-s"]))
        ```
  
**Changes I made:**

1. Fixed a crash in _check_where. DuckDB's parse tree sometimes stores type fields as nested objects rather than strings, which caused .upper() to raise an AttributeError on queries with a WHERE clause. I added a _s() helper to safely handle non-string values.

2. Changed NULL_FILTER for non-negated predicates (=, IN) from a warning to an info message because excluding NULL values is often intentional, and the warning was triggered by correct queries. Negated predicates (!=, NOT IN) remain errors.

3. The model could not run the [Evaluate] step and said so, so I ran the function myself on my Phase 1 and Phase 2 queries and reported what it caught and missed.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.